# Payerne Raman lidar + radiosonde vs KENDA-CH1 801/802 — profile frames

Produces one PNG per valid time comparing the temperature / water-vapour / RH profile above Payerne
from the **Raman lidar (RALMO)**, the **radiosonde** where one was launched that hour, and
experiments **801** and **802** — separately for **`iaf`** (post-IAU analysis) and **`lff`** (+1 h
first guess), so the sequences can be animated and stepped case by case.

    figures/ramanlidar_iaf_frames/ramanlidar_iaf_<YYYYMMDDHH>.png
    figures/ramanlidar_lff_frames/ramanlidar_lff_<YYYYMMDDHH>.png

Step through them with the repo's viewer:

    python3 scripts/frame_viewer.py figures/ramanlidar_iaf_frames

## Raman lidar

`/users/daniel/work/kenda/ramanlidar/YYYYMMDD_06610_data.txt`, 30-min cadence, `level` in m **asl**.
The columns are bare DWH parameter IDs which are *not* in `parameter_dictionary.txt` (that file is
only a DWH-id → fieldextra-name crosswalk for the ATAB format, and `ramanlidar` does not support
ATAB). Identities were established against the Payerne radiosonde at matching `termin` (24 launches,
levels below 6 km):

| ID | quantity | unit | evidence |
|----|----------|------|----------|
| `3147` | temperature | K | bias vs sonde −0.21 K, RMS 1.70 K |
| `4908` | uncertainty of `3147` | K | sonde-error RMS rises 1.29 → 4.51 K as `4908` goes 0.7 → 5.1 |
| `4919` | water vapour | g/kg | median ratio to sonde 0.90 |
| `4906` | uncertainty of `4919` | g/kg | sonde-error IQR rises 0.33 → 0.92 as `4906` goes 0.0 → 1.0 |

Two working assumptions: `termin` is treated as **instantaneous** (RALMO actually integrates ~30 min),
and `4919` is treated as the same quantity as ICON `QV` (mixing ratio vs specific humidity differ by
~1 %, far below the ~10 % lidar-sonde spread).

**The lidar is ~10 % dry against the radiosonde**, and no correction is applied. Having the sonde on
the same axes makes that offset directly visible on the ~6 frames per window that carry both — which
is the main reason to want it there. It is identical in both experiments, so 801-vs-802 differences
within a frame are unaffected.

## Radiosonde

`/scratch/mch/jdelbeke/soundings/output/`, **00 and 12 UTC only**. In the 801/802 window there are 14
launches, of which 6 fall on an hour that also has a lidar profile. Humidity is derived from the dew
point; details and the two file traps are in section 2.

## Model column

Mean of the **6 nearest ICON-CH1 cells** to the station (46.8116 N, 6.9424 E): 746375 (0.31 km),
746374, 746368, 746365, 746372, 746370, all ≤ 1.33 km. Their mean `HSURF` is 488.6 m vs the station's
490 m. Model levels 79–80 (centres ~504 and ~484 m asl) lie below the first lidar gate at 551 m and
are not observable by the lidar — the sonde does reach them.

In [ ]:
# ==========================================================
# CONFIG + IMPORTS
# ==========================================================
import os

# The uenv pairs definitions.edzw 2.47.0 with libeccodes 2.47.3, and ecCodes prints a version banner
# to stderr on *every* GRIB file it opens — hundreds of times over the build loop, which in Jupyter
# looks like a wall of errors. This switches the check off at source. Must be set before
# earthkit.data is imported. Verified to leave decoded values unchanged.
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

# eccodes must be imported before earthkit.data so the native libraries resolve correctly
import eccodes  # noqa: F401,E402
import earthkit.data as ekd  # noqa: E402

import collections  # noqa: E402
import warnings  # noqa: E402
from pathlib import Path  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import xarray as xr  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402

# ----------------------------------------------------------
# Raman lidar (RALMO)
# ----------------------------------------------------------
LIDAR_DIR = Path("/users/daniel/work/kenda/ramanlidar")
STATION   = "06610"
MISSING   = 10_000_000

# DWH parameter ids -> our names, in file column order (4919, 4906, 3147, 4908)
LIDAR_COLS = ["wmo_id", "termin", "lat", "lon", "elev", "stn_name", "int_ind",
              "track_type", "prof_type", "type", "level",
              "qv", "qv_unc", "T", "T_unc"]

# Deliberately loose: the reported uncertainty predicts the actual error, so it is more useful to
# keep the data and draw the uncertainty band than to discard levels here. A strict T cut of 2.0
# empties some profiles entirely.
MAX_T_UNC  = 3.0    # K
MAX_QV_UNC = 1.5    # g/kg

# Hard ceiling for humidity: above ~5-6 km the retrieval is noise AND its uncertainty rounds to 0.0,
# so MAX_QV_UNC cannot catch it. Without this the noise is drawn as if it were valid humidity.
# Measured: above 6000 m, 27.6 % of otherwise-surviving points are physically impossible.
MAX_QV_HEIGHT_M = 6000.0   # m asl

# Physical plausibility ceiling on the lidar humidity, as implied RH over water. The reported
# uncertainty does not catch impossible supersaturation, and the height cap only removes the bulk of
# it, so this is a separate and independent filter. 120 % leaves a wide margin: the model peaks at
# 100.5 % and the sonde at 100.8 %, so nothing legitimate sits near this threshold.
MAX_QV_RH = 120.0   # %

# Maximum gap the level-matching fallback may interpolate across. Without a limit the fallback
# invented 437 temperature values per 81 slots spanning a median gap of 3840 m (max 9150 m) — data
# drawn as observation where the instrument saw nothing. 150 m allows bridging ~5 lidar gates; in
# practice almost nothing legitimate needs it, because 30 m gates always populate a 38-44 m layer
# when the lidar reported at all.
MAX_INTERP_GAP_M = 150.0   # m

# ----------------------------------------------------------
# Radiosonde — same DWH export layout, different parameter ids
# ----------------------------------------------------------
SOUNDING_DIR = Path("/scratch/mch/jdelbeke/soundings/output")

# 745 = T [degC], 747 = dew point [degC], 748 = wind speed, 743 = wind direction,
# 742 = height [m asl], 744 = pressure [hPa]
SND_COLS = ["wmo_id", "termin", "lat", "lon", "elev", "stn_name", "int_ind", "nat_abbr",
            "track_type", "prof_type", "type", "level",
            "T_C", "Td_C", "ff", "dd", "height", "p"]

# ----------------------------------------------------------
# Model
# ----------------------------------------------------------
EXPS     = ["801", "802"]
EXP_ROOT = Path("/store_new/mch/msopr/jdelbeke/ICON_TST")

# file kind -> (subdirectory, filename prefix). Both carry T/QV/P (80 lev) and HHL (81).
KINDS = {
    "iaf": ("ANA25/det", "iaf"),   # post-IAU analysis
    "lff": ("FG25/det",  "lff"),   # +1 h first guess from the previous cycle
}

PAYERNE_CELLS = np.array([746375, 746374, 746368, 746365, 746372, 746370])
STATION_ELEV  = 490.0   # m asl
MODEL_VARS    = ["T", "QV", "P"]
N_FULL_LEV    = 80

# ----------------------------------------------------------
# Output
# ----------------------------------------------------------
CACHE_DIR = Path("/scratch/mch/jdelbeke")
FIG_DIR   = Path("figures")
PLOT_TOP_M = 5000.0     # top of the plotted range, m above model ground

CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)

# netCDF4 in this uenv is built against a different numpy ABI; the import warning is benign.
warnings.filterwarnings("ignore", message="numpy.ndarray size changed")

## 1. Nearest model cells

Hard-coded above so the notebook does not re-read the 1.15 M-cell grid every run. This cell
re-derives and checks them.

In [2]:
# ==========================================================
# STANDALONE CELL — re-derive the nearest ICON-CH1 cells
# ==========================================================
GRID = "/oprusers/osm/opr.inn/config/resources/grid_descriptions/icon_grid_0001_R19B08_mch.nc"
PAY_LAT, PAY_LON = 46.8116, 6.9424

_g = xr.open_dataset(GRID)
_lat, _lon = np.degrees(_g["clat"].values), np.degrees(_g["clon"].values)
_d = 6371.0 * np.hypot(np.radians(_lon - PAY_LON) * np.cos(np.radians(PAY_LAT)),
                       np.radians(_lat - PAY_LAT))
_near = np.argsort(_d)[:6]

print(f"{_lat.size} cells; 6 nearest to Payerne:")
for k in _near:
    print(f"  idx {k:7d}   {_d[k]:.3f} km   lat {_lat[k]:.5f}  lon {_lon[k]:.5f}")
print("\nmatches PAYERNE_CELLS:", np.array_equal(np.sort(_near), np.sort(PAYERNE_CELLS)))
_g.close()

1147980 cells; 6 nearest to Payerne:
  idx  746375   0.314 km   lat 46.81284  lon 6.93869
  idx  746374   0.733 km   lat 46.81803  lon 6.94453
  idx  746368   0.749 km   lat 46.80487  lon 6.94242
  idx  746365   1.318 km   lat 46.81384  lon 6.92540
  idx  746372   1.320 km   lat 46.81703  lon 6.95783
  idx  746370   1.329 km   lat 46.80387  lon 6.95571

matches PAYERNE_CELLS: True


## 2. Observation readers

Both observation sources use the same pipe-delimited DWH layout: header on line 2, data from line 4,
`10000000` as a **per-field** sentinel. QC masks values rather than dropping rows, so the T and qv
masks stay independent — they fail at different heights in both instruments.

**Raman lidar** — 30-min cadence, `level` already in m asl.

**Radiosonde** — `/scratch/mch/jdelbeke/soundings/output/`, launches at **00 and 12 UTC only**, so
roughly 2 of 24 hours can carry one. Two traps handled here:

- the `level == -100` row is a per-launch header carrying a fake 85 m height and 1000 hPa pressure
  with every measured field set to the sentinel — left in, it puts a spurious level at the base of
  the profile;
- there is **no humidity column**. Saturation vapour pressure evaluated *at the dew point* is the
  actual vapour pressure, which gives the mixing ratio. This uses the same saturation formula as the
  model and lidar sides — deriving it the textbook way instead would make the quantities subtly
  different and turn a formula mismatch into an apparent model error.

The sonde reader returns the same columns as the lidar reader (`time / level / T / qv / p`), so the
identical level-matching operator handles both. Unlike the lidar, the sonde **does** measure
pressure, so its RH uses its own.

Note the sonde is not an instantaneous profile either: it takes tens of minutes to climb and drifts
horizontally, so at 5 km it is neither at the launch time nor above the station.

In [ ]:
# ==========================================================
# Observation readers: Raman lidar and radiosonde
# ==========================================================
def standard_pressure(z_m):
    """ICAO standard-atmosphere pressure [hPa] at geometric height z [m asl].

    Used *only* for the QC saturation check below, so that filtering the observations stays
    independent of the model. A couple of percent of pressure error is irrelevant against a
    120 % RH threshold.
    """
    return 1013.25 * (1.0 - 2.25577e-5 * np.asarray(z_m, dtype=float)) ** 5.25588


def implied_rh_over_water(T_K, qv_gkg, z_m):
    """RH [%] over water implied by a (T, qv) pair, on standard-atmosphere pressure."""
    q = np.asarray(qv_gkg, dtype=float) / 1000.0
    p = standard_pressure(z_m)
    e = q * p / (0.622 + 0.378 * q)
    return 100.0 * e / saturation_vapour_pressure(T_K)


def read_lidar_day(day, apply_qc=True):
    """Read one daily RALMO file. `day` is 'YYYYMMDD'."""
    path = LIDAR_DIR / f"{day}_{STATION}_data.txt"
    if not path.exists():
        return None
    df = pd.read_csv(path, sep="|", skiprows=3, header=None, names=LIDAR_COLS, engine="c")
    if df.empty:                      # header-only: no data, or a failed DWH retrieval
        return None

    df = df.replace(MISSING, np.nan)
    df["time"] = pd.to_datetime(df["termin"].astype("int64").astype(str), format="%Y%m%d%H%M%S")

    if apply_qc:
        # Computed BEFORE the temperature QC and from the raw columns on purpose: a temperature with
        # a large reported uncertainty is still good enough for a plausibility check, and this
        # filter must not depend on whether T survived its own threshold.
        rh_implied = implied_rh_over_water(df["T"], df["qv"], df["level"])

        df.loc[~(df["T_unc"] <= MAX_T_UNC), "T"] = np.nan
        df.loc[~(df["qv_unc"] <= MAX_QV_UNC), "qv"] = np.nan
        df.loc[df["level"] > MAX_QV_HEIGHT_M, "qv"] = np.nan

        # Physically impossible supersaturation, which the reported uncertainty does NOT catch:
        # median qv_unc of the offenders is 0.4 g/kg, well inside MAX_QV_UNC. They are not confined
        # to high levels either — over the whole archive they are 0.14 % of points at 0.5-1.5 km
        # rising to 0.61 % at 2.5-3.5 km, so tightening MAX_QV_HEIGHT_M does not remove them
        # (cap 6000 -> 4000 m drops only 2048 -> 1797 bad points while discarding 20 % of good
        # data). Worst case seen: 9.81 g/kg at 249.6 K, i.e. RH ~ 870 %.
        df.loc[rh_implied > MAX_QV_RH, "qv"] = np.nan

    return df[["time", "level", "T", "T_unc", "qv", "qv_unc"]].sort_values(["time", "level"])


def load_lidar(days, apply_qc=True):
    parts = [d for d in (read_lidar_day(day, apply_qc) for day in days) if d is not None]
    return pd.concat(parts, ignore_index=True) if parts else None


def read_sounding_day(day):
    """Read one daily Payerne radiosonde file (00 and 12 UTC launches stacked).

    Returns the same shape as the lidar reader — time / level [m asl] / T [K] / qv [g/kg] / p [hPa]
    — so the identical level-matching operator works on both.
    """
    path = SOUNDING_DIR / f"{day}_{STATION}_data.txt"
    if not path.exists():
        return None
    df = pd.read_csv(path, sep="|", skiprows=3, header=None, names=SND_COLS, engine="c")
    if df.empty:
        return None
    df = df.replace(MISSING, np.nan)

    # level == -100 is a per-launch header row carrying a fake height of 85 m and pressure of
    # 1000 hPa with every measured field set to the sentinel. Left in, it puts a spurious level at
    # the base of the profile.
    df = df[df["level"] >= 0]
    df = df[np.isfinite(df["height"])]
    if df.empty:
        return None

    df["time"] = pd.to_datetime(df["termin"].astype("int64").astype(str), format="%Y%m%d%H%M%S")
    df["T"] = df["T_C"] + 273.15

    # There is no humidity column: saturation vapour pressure evaluated *at the dew point* is the
    # actual vapour pressure, which gives the mixing ratio. Uses the same saturation formula as the
    # model side — deriving it the textbook way instead would make the two subtly different
    # quantities and turn a formula mismatch into an apparent model error.
    # The hygrometer drops out where the thermometer still reports, so qv goes NaN there and T stays.
    # No plausibility filter is needed here: qv is derived from a dew point, so it cannot exceed
    # saturation by construction (verified: max implied RH over the window is 100.8 %).
    e = saturation_vapour_pressure(df["Td_C"] + 273.15)
    df["qv"] = 622.0 * e / (df["p"] - e)

    return (df[["time", "height", "T", "qv", "p"]]
            .rename(columns={"height": "level"})
            .sort_values(["time", "level"]))


def load_soundings(days):
    parts = [d for d in (read_sounding_day(day) for day in days) if d is not None]
    return pd.concat(parts, ignore_index=True) if parts else None

## 3. Model column extraction

Each file is indexed once, then only the 6 target cells are pulled out of each level. `HHL` (81 half
levels) is read once per experiment, since the level geometry is static — and it has to be read
rather than assumed, because ICON's levels follow the terrain.

Note `earthkit`'s `fieldlist.sel(shortName=...)` returns an **empty** result on these files — no
index is built automatically — so the `shortName → [Field]` mapping is assembled by hand.

The repeated ecCodes version banner is switched off with `ECCODES_VERSION_CHECK_OFF=1` in the config
cell, which is why there is no stderr plumbing here.

In [ ]:
# ==========================================================
# Model column extraction
# ==========================================================
def model_path(kind, exp, t):
    subdir, prefix = KINDS[kind]
    return EXP_ROOT / exp / subdir / f"{prefix}{t:%Y%m%d%H}"


def extract_column(path, cells=PAYERNE_CELLS, varnames=MODEL_VARS, with_geometry=False):
    """Mean over `cells` of each level. Keeps ICON ordering: index 0 = level 1 = model top."""
    wanted = set(varnames) | ({"HHL", "HSURF"} if with_geometry else set())

    # fieldlist.sel(shortName=...) silently returns nothing on these files — no index is built
    # automatically — so assemble the shortName -> [Field] mapping by hand.
    by = collections.defaultdict(list)
    for f in ekd.from_source("file", str(path)).to_fieldlist():
        sn = f.metadata("shortName")
        if sn in wanted:
            by[sn].append(f)

    out = {}
    for name in varnames:
        fields = sorted(by[name], key=lambda f: f.metadata("level"))
        if len(fields) != N_FULL_LEV:
            raise ValueError(f"{name}: expected {N_FULL_LEV} levels in {path}, got {len(fields)}")
        out[name] = np.array([f.to_numpy().ravel()[cells].mean() for f in fields])

    if with_geometry:
        hhl_f = sorted(by["HHL"], key=lambda f: f.metadata("level"))
        out["HHL"]   = np.array([f.to_numpy().ravel()[cells].mean() for f in hhl_f])
        out["HSURF"] = float(by["HSURF"][0].to_numpy().ravel()[cells].mean())
    return out


def layer_edges(hhl):
    """Full-level centres and layer top/bottom edges [m asl] from the 81 half levels."""
    top, bot = hhl[:-1], hhl[1:]      # HHL level 1 = model top, level 81 = ground
    return 0.5 * (top + bot), top, bot

## 4. Putting the observations on the model levels

Lidar gates are 30 m apart and sonde levels ~5 m; the model layers are 38 m thick at the first
observed level and ~210 m at 5.8 km. The rule: **average the observation levels inside a model
layer**; if none fall inside, **interpolate to the layer centre — but only across a gap of at most
`MAX_INTERP_GAP_M`**, otherwise leave it empty.

That gap limit was added after an audit. Without it the fallback produced 437 temperature values per
81 slots bridging a **median gap of 3840 m** (max 9150 m), i.e. drawing invented observations across
kilometres of missing profile. Tightening the tolerance to 60 m leaves exactly the values that came
from genuine averaging — which shows the near-ground case the fallback was originally justified by
never actually happens here: with 30 m gates and 38-44 m layers, a layer always contains a gate
whenever the lidar reported at all. The same limit removed an implausible 178.7 % RH that no per-gate
filter could catch, because the offending humidity had been interpolated across a 330 m hole.

Model levels 79-80 (centres ~504 and ~484 m asl) lie below the lidar's first gate at 551 m, so the
lidar can never populate them. The sonde does reach them.

In [ ]:
# ==========================================================
# Observation -> model level operator, and RH
# ==========================================================
def obs_to_model_levels(obs_level, obs_value, centres, top, bot):
    """Map one observed profile onto the model full levels. Returns (values, n_gates).

    Average the observation levels that fall inside a model layer. If none do, interpolate to the
    layer centre, but ONLY across a gap of at most MAX_INTERP_GAP_M; otherwise leave it empty.

    That limit is not cosmetic. Measured over the 81 iaf slots without it, the fallback produced 437
    temperature values bridging a *median gap of 3840 m* (max 9150 m) — inventing observations across
    kilometres of missing profile and drawing them as data. A 60 m tolerance leaves exactly the
    values that came from real averaging, which shows the near-ground case the fallback was meant for
    never actually occurs here: the lidar's 30 m gates always populate a 38-44 m model layer whenever
    the instrument reported at all. It also removed an implausible 178.7 % RH that no per-gate filter
    could catch, because the offending humidity had been interpolated across a 330 m hole.
    """
    ok = np.isfinite(obs_value) & np.isfinite(obs_level)
    lev = np.asarray(obs_level, dtype=float)[ok]
    val = np.asarray(obs_value, dtype=float)[ok]

    out = np.full(len(centres), np.nan)
    cnt = np.zeros(len(centres), dtype=int)
    if lev.size == 0:
        return out, cnt

    order = np.argsort(lev)
    lev, val = lev[order], val[order]

    for k in range(len(centres)):
        inside = (lev >= bot[k]) & (lev < top[k])
        n = int(inside.sum())
        cnt[k] = n
        if n:
            out[k] = val[inside].mean()
            continue
        # nothing inside the layer: interpolate only if real data sits close on BOTH sides
        c = centres[k]
        below, above = lev[lev <= c], lev[lev >= c]
        if below.size and above.size \
                and (c - below[-1]) <= MAX_INTERP_GAP_M \
                and (above[0] - c) <= MAX_INTERP_GAP_M:
            out[k] = np.interp(c, lev, val)
    return out, cnt


def saturation_vapour_pressure(T_K):
    """Magnus, over water, hPa. Used on both sides so the comparison stays fair."""
    Tc = np.asarray(T_K) - 273.15
    return 6.112 * np.exp(17.62 * Tc / (243.12 + Tc))


def relative_humidity(T_K, q_gkg, p_hPa):
    """RH [%]. The lidar has no pressure channel, so its RH uses the model's pressure.

    Over water at all heights on every curve, so the lines are comparable — but that means absolute
    values below freezing are not the WMO ice convention. Feeding a mixing ratio to a formula that
    wants specific humidity costs ~0.5 % in RH, far below the observation error.
    """
    q = np.asarray(q_gkg) / 1000.0
    e = q * np.asarray(p_hPa) / (0.622 + 0.378 * q)
    return 100.0 * e / saturation_vapour_pressure(T_K)

## 5. Build the matched datasets

One dataset per file kind, cached to netCDF. `iaf` covers 2025-10-04 00 → 10-10 00 and `lff`
2025-10-04 01 → 10-10 01, both hourly; only hours with a lidar profile on the same hour are kept.

Roughly 12 s per file, ~2 files per slot per kind, so about 20 min per kind on the first run and
instant thereafter. Set `FORCE_REBUILD = True` to regenerate, `MAX_TIMES` for a quick test.

In [ ]:
# ==========================================================
# Build (or load) the matched datasets
# ==========================================================
from concurrent.futures import ProcessPoolExecutor

FORCE_REBUILD = False
MAX_TIMES     = None        # e.g. 5 for a quick test run

# Reading one file takes ~5-8 s, almost all of it waiting on /store_new rather than on Python, so
# the fix for a slow build is to overlap the I/O. Profiling showed the earthkit index pass and a
# hand-rolled low-level eccodes reader are the same speed once the file is in cache, so there is
# nothing to gain from a cleverer reader. 16 workers is polite to a shared filesystem; the node has
# far more cores than that.
N_WORKERS = min(16, os.cpu_count() or 1)


def provenance_tag():
    """Everything that changes the contents of a cache. Stored in the file and checked on load.

    Without this a cache written under different QC settings is silently reused, which is exactly
    how the iaf and lff sequences ended up built from different thresholds once already.
    N_WORKERS is deliberately absent: it changes speed, not results.
    """
    return (f"exps={','.join(EXPS)}; cells={','.join(map(str, PAYERNE_CELLS))}; "
            f"T_unc<={MAX_T_UNC}; qv_unc<={MAX_QV_UNC}; qv_top<={MAX_QV_HEIGHT_M}; qv_rh<={MAX_QV_RH}; interp_gap<={MAX_INTERP_GAP_M}; "
            f"sounding=on")


def available_times(kind, exp):
    subdir, prefix = KINDS[kind]
    d = EXP_ROOT / exp / subdir
    return sorted(pd.to_datetime(p.name[len(prefix):], format="%Y%m%d%H")
                  for p in d.glob(f"{prefix}" + "?" * 10))


def _slot_job(job):
    """One time slot, both experiments. Top level so ProcessPoolExecutor can pickle it."""
    kind, exps, t = job
    return {exp: extract_column(model_path(kind, exp, t)) for exp in exps}


def build_matched(kind):
    times = sorted(set.intersection(*(set(available_times(kind, e)) for e in EXPS)))
    days = sorted({t.strftime("%Y%m%d") for t in times})

    obs = load_lidar(days)
    if obs is None:
        raise RuntimeError(f"no lidar data overlapping the {kind} window")
    snd = load_soundings(days)          # may be None; soundings are only at 00 and 12 UTC

    # instantaneous assumption: match the lidar termin exactly to the model valid time
    have = set(obs["time"].unique())
    times = [t for t in times if np.datetime64(t) in have]
    if MAX_TIMES:
        times = times[:MAX_TIMES]
    print(f"[{kind}] {len(times)} matched hourly slots, {N_WORKERS} workers", flush=True)

    geom = extract_column(model_path(kind, EXPS[0], times[0]), varnames=[], with_geometry=True)
    centres, top, bot = layer_edges(geom["HHL"])

    nt, ne, nl = len(times), len(EXPS), N_FULL_LEV
    mod  = {v: np.full((ne, nt, nl), np.nan) for v in MODEL_VARS}
    o_T  = np.full((nt, nl), np.nan);  o_qv = np.full((nt, nl), np.nan)
    u_T  = np.full((nt, nl), np.nan);  u_qv = np.full((nt, nl), np.nan)
    s_T  = np.full((nt, nl), np.nan);  s_qv = np.full((nt, nl), np.nan)
    s_p  = np.full((nt, nl), np.nan)

    # --- observations: pure pandas, fast, stays in this process
    for i, t in enumerate(times):
        p = obs[obs["time"] == t]
        o_T[i],  _ = obs_to_model_levels(p["level"], p["T"],  centres, top, bot)
        o_qv[i], _ = obs_to_model_levels(p["level"], p["qv"], centres, top, bot)
        u_T[i],  _ = obs_to_model_levels(p["level"], p["T_unc"].where(p["T"].notna()),
                                         centres, top, bot)
        u_qv[i], _ = obs_to_model_levels(p["level"], p["qv_unc"].where(p["qv"].notna()),
                                         centres, top, bot)

        # radiosonde, where a launch coincides with this hour (00/12 UTC only)
        if snd is not None:
            g = snd[snd["time"] == t]
            if len(g):
                s_T[i],  _ = obs_to_model_levels(g["level"], g["T"],  centres, top, bot)
                s_qv[i], _ = obs_to_model_levels(g["level"], g["qv"], centres, top, bot)
                s_p[i],  _ = obs_to_model_levels(g["level"], g["p"],  centres, top, bot)

    # --- model: the slow part, parallel over time slots.
    # pool.map preserves input order, so enumerate() still lines results up with `times`.
    jobs = [(kind, EXPS, t) for t in times]
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        for i, res in enumerate(pool.map(_slot_job, jobs)):
            for j, exp in enumerate(EXPS):
                for v in MODEL_VARS:
                    mod[v][j, i] = res[exp][v]
            if (i + 1) % 10 == 0 or i == nt - 1:
                print(f"  [{kind}] {i + 1}/{nt}  {times[i]}", flush=True)

    n_empty = int((~np.isfinite(o_T) & ~np.isfinite(o_qv)).all(axis=1).sum())
    n_snd = int(np.isfinite(s_T).any(axis=1).sum())
    print(f"[{kind}] slots with no usable T: {int((~np.isfinite(o_T)).all(axis=1).sum())}/{nt}; "
          f"no usable qv: {int((~np.isfinite(o_qv)).all(axis=1).sum())}/{nt}; "
          f"nothing at all: {n_empty}/{nt}")
    print(f"[{kind}] slots that also have a radiosonde: {n_snd}/{nt}")

    ds = xr.Dataset(
        {
            "T_mod":  (("exp", "time", "level"), mod["T"]),
            "qv_mod": (("exp", "time", "level"), mod["QV"] * 1000.0),   # kg/kg -> g/kg
            "p_mod":  (("exp", "time", "level"), mod["P"] / 100.0),     # Pa -> hPa
            "T_obs":  (("time", "level"), o_T),
            "qv_obs": (("time", "level"), o_qv),
            "T_obs_unc":  (("time", "level"), u_T),
            "qv_obs_unc": (("time", "level"), u_qv),
            "T_snd":  (("time", "level"), s_T),
            "qv_snd": (("time", "level"), s_qv),
            "p_snd":  (("time", "level"), s_p),
            "height":     ("level", centres),
            "height_agl": ("level", centres - geom["HSURF"]),
        },
        coords={"exp": EXPS, "time": times, "level": np.arange(1, nl + 1)},
        attrs={"kind": kind, "hsurf_model_m": geom["HSURF"], "station_elev_m": STATION_ELEV,
               "provenance": provenance_tag(),
               "note": "obs assumed instantaneous at termin; lidar qv ~10% dry vs radiosonde"},
    )
    return ds


DS = {}
for kind in KINDS:
    cache = CACHE_DIR / f"ramanlidar_{kind}_matched.nc"

    reuse = None
    if cache.exists() and not FORCE_REBUILD:
        candidate = xr.open_dataset(cache)
        if candidate.attrs.get("provenance") == provenance_tag():
            reuse = candidate
        else:
            candidate.close()
            print(f"[{kind}] cache is stale, rebuilding\n"
                  f"      cached: {candidate.attrs.get('provenance', '(none recorded)')}\n"
                  f"      wanted: {provenance_tag()}")

    if reuse is not None:
        DS[kind] = reuse
        print(f"[{kind}] loaded cache: {cache}  ({DS[kind].sizes['time']} slots)")
    else:
        DS[kind] = build_matched(kind)
        DS[kind].to_netcdf(cache)
        print(f"[{kind}] wrote cache: {cache}  ({DS[kind].sizes['time']} slots)")

## 6. Frames

One PNG per valid time, per file kind. Four things per panel:

| | style |
|---|---|
| **RALMO** | black, dots and line, with a grey band showing its own reported uncertainty |
| **radiosonde** | green dashed — only at 00/12 UTC, so most frames will not have it |
| **801** | blue |
| **802** | orange |

Axis limits are fixed across every frame **and** shared between `iaf` and `lff`, so the sequences
animate without the axes jumping and can be compared against each other. The sounding is included in
that calculation so it cannot land off-scale.

Frames where the *lidar* has nothing usable are skipped and counted — a frame is keyed to the lidar,
which means the 8 sounding launches falling on hours with no lidar profile get no frame. Say if you
want those drawn too.

The title marks whether a sounding is present, so it is obvious while scrubbing.

In [ ]:
# ==========================================================
# Frame generation
# ==========================================================
OVERWRITE_FRAMES = False        # True to redraw frames that already exist
EXP_COLOURS = {"801": "C0", "802": "C1"}
SND_COLOUR  = "C2"              # radiosonde: green, dashed — an observation, but not the lidar


def snd_rh(ds, i=slice(None)):
    """Sounding RH, using the sonde's *own* pressure (unlike the lidar, it measures pressure)."""
    return relative_humidity(ds["T_snd"].values[i], ds["qv_snd"].values[i], ds["p_snd"].values[i])


def frame_limits(datasets):
    """Fixed axis limits, shared across every frame AND across iaf/lff.

    Sharing across kinds matters: flipping between the two animations only means something if the
    axes are identical in both. The sounding is included so it cannot land off-scale.
    """
    def collect(pick):
        vals = []
        for ds in datasets:
            z_ok = ds["height_agl"].values <= PLOT_TOP_M
            for a in pick(ds):
                vals.append(np.asarray(a)[..., z_ok].ravel())
        v = np.concatenate(vals)
        return v[np.isfinite(v)]

    def rng(pick, pad_frac=0.05, floor=None):
        v = collect(pick)
        lo, hi = np.percentile(v, [0.5, 99.5])
        pad = pad_frac * (hi - lo)
        lo, hi = lo - pad, hi + pad
        return (max(lo, floor) if floor is not None else lo), hi

    def rh_of(ds):
        out = [relative_humidity(ds["T_mod"].values[j], ds["qv_mod"].values[j],
                                 ds["p_mod"].values[j]) for j in range(ds.sizes["exp"])]
        out.append(relative_humidity(ds["T_obs"].values, ds["qv_obs"].values,
                                     ds["p_mod"].values[0]))
        out.append(snd_rh(ds))
        return out

    return {
        "T":  rng(lambda d: [d["T_mod"].values, d["T_obs"].values, d["T_snd"].values]),
        "qv": rng(lambda d: [d["qv_mod"].values, d["qv_obs"].values, d["qv_snd"].values],
                  floor=0.0),
        "rh": rng(rh_of, floor=0.0),
    }


def draw_frame(ds, i, lims, kind, outpath):
    z = ds["height_agl"].values
    t = pd.Timestamp(ds["time"].values[i])

    fig, ax = plt.subplots(1, 3, figsize=(13, 7), sharey=True)

    # --- lidar, with its reported uncertainty as a band
    for a, var, unc in ((ax[0], "T_obs", "T_obs_unc"), (ax[1], "qv_obs", "qv_obs_unc")):
        v, u = ds[var].values[i], ds[unc].values[i]
        a.fill_betweenx(z, v - u, v + u, color="0.7", alpha=0.5, lw=0, zorder=2)
        a.plot(v, z, "k.-", ms=3, lw=1.2, label="RALMO", zorder=4)

    rh_obs = relative_humidity(ds["T_obs"].values[i], ds["qv_obs"].values[i],
                               ds["p_mod"].values[0, i])
    ax[2].plot(rh_obs, z, "k.-", ms=3, lw=1.2, label="RALMO (+ model p)", zorder=4)

    # --- radiosonde, only present at 00/12 UTC
    has_snd = bool(np.isfinite(ds["T_snd"].values[i]).any()
                   or np.isfinite(ds["qv_snd"].values[i]).any())
    if has_snd:
        style = dict(color=SND_COLOUR, ls="--", lw=1.5, zorder=3)
        ax[0].plot(ds["T_snd"].values[i],  z, label="sounding", **style)
        ax[1].plot(ds["qv_snd"].values[i], z, label="sounding", **style)
        ax[2].plot(snd_rh(ds, i), z, label="sounding (own p)", **style)

    # --- model
    for j, exp in enumerate(ds["exp"].values):
        c = EXP_COLOURS.get(str(exp), f"C{j + 3}")
        ax[0].plot(ds["T_mod"].values[j, i],  z, "-", color=c, lw=1.4, label=exp)
        ax[1].plot(ds["qv_mod"].values[j, i], z, "-", color=c, lw=1.4, label=exp)
        ax[2].plot(relative_humidity(ds["T_mod"].values[j, i], ds["qv_mod"].values[j, i],
                                     ds["p_mod"].values[j, i]), z, "-", color=c, lw=1.4, label=exp)

    ax[2].axvline(100, color="0.6", lw=0.8, zorder=1)

    ax[0].set_xlabel("T [K]");      ax[0].set_xlim(*lims["T"])
    ax[1].set_xlabel("qv [g/kg]");  ax[1].set_xlim(*lims["qv"])
    ax[2].set_xlabel("RH [%]");     ax[2].set_xlim(*lims["rh"])
    ax[0].set_ylabel("height above model ground [m]")
    # explicit and idempotent: invert_yaxis() toggles and misfires under sharey
    ax[0].set_ylim(0, PLOT_TOP_M)
    for a in ax:
        a.grid(alpha=0.3)
        a.legend(fontsize=8, loc="upper right")

    n_T  = int(np.isfinite(ds["T_obs"].values[i]).sum())
    n_qv = int(np.isfinite(ds["qv_obs"].values[i]).sum())
    fig.suptitle(f"Payerne RALMO{' + sounding' if has_snd else ''} vs {kind}"
                 f"  —  {t:%Y-%m-%d %H:%M} UTC   (RALMO levels: T {n_T}, qv {n_qv})")
    fig.tight_layout()
    fig.savefig(outpath, dpi=120)
    plt.close(fig)


LIMS = frame_limits(list(DS.values()))       # one set of axes for iaf and lff alike
print(f"shared limits  T {LIMS['T'][0]:.1f}-{LIMS['T'][1]:.1f} K   "
      f"qv {LIMS['qv'][0]:.2f}-{LIMS['qv'][1]:.2f} g/kg   "
      f"RH {LIMS['rh'][0]:.0f}-{LIMS['rh'][1]:.0f} %")

# Frames inherit the cache's provenance *plus* the shared axis limits. If either changed, existing
# PNGs are stale and must be redrawn rather than skipped.
FRAME_TAG = (provenance_tag() + f"; limits={LIMS['T']}, {LIMS['qv']}, {LIMS['rh']}"
             f"; top={PLOT_TOP_M}")

for kind, ds in DS.items():
    outdir = FIG_DIR / f"ramanlidar_{kind}_frames"
    outdir.mkdir(parents=True, exist_ok=True)

    tagfile = outdir / "provenance.txt"
    stale = tagfile.read_text() != FRAME_TAG if tagfile.exists() else any(outdir.glob("*.png"))
    if stale:
        print(f"[{kind}] existing frames were made with different settings — redrawing all")

    written = skipped_empty = skipped_exist = with_snd = 0

    for i in range(ds.sizes["time"]):
        if not (np.isfinite(ds["T_obs"].values[i]).any()
                or np.isfinite(ds["qv_obs"].values[i]).any()):
            skipped_empty += 1
            continue
        t = pd.Timestamp(ds["time"].values[i])
        out = outdir / f"ramanlidar_{kind}_{t:%Y%m%d%H}.png"
        if np.isfinite(ds["T_snd"].values[i]).any():
            with_snd += 1
        if out.exists() and not (OVERWRITE_FRAMES or stale):
            skipped_exist += 1
            continue
        draw_frame(ds, i, LIMS, kind, out)
        written += 1

    tagfile.write_text(FRAME_TAG)
    print(f"[{kind}] {written} frames written, {skipped_exist} already present, "
          f"{skipped_empty} skipped (no usable observation)  ->  {outdir}")
    print(f"      {with_snd} of those frames also show a radiosonde")

## 7. Stepping through them

```bash
python3 scripts/frame_viewer.py figures/ramanlidar_iaf_frames
python3 scripts/frame_viewer.py figures/ramanlidar_lff_frames
```

Arrow keys step, the slider scrubs, `play` animates. Stdlib only, so the plain login-node `python3`
runs it — no uenv or notebook kernel needed.

Frame filenames are `ramanlidar_<kind>_<YYYYMMDDHH>.png`, so they sort chronologically and feed
straight into `ffmpeg`/`convert` if you want a GIF or MP4 instead.

In [ ]:
# ==========================================================
# STANDALONE CELL — what got produced
# ==========================================================
for kind in KINDS:
    outdir = FIG_DIR / f"ramanlidar_{kind}_frames"
    frames = sorted(outdir.glob(f"ramanlidar_{kind}_*.png"))
    if not frames:
        print(f"[{kind}] no frames in {outdir}")
        continue
    stamps = [pd.to_datetime(f.stem.split("_")[-1], format="%Y%m%d%H") for f in frames]
    size_mb = sum(f.stat().st_size for f in frames) / 1e6
    print(f"[{kind}] {len(frames)} frames, {size_mb:.1f} MB, "
          f"{stamps[0]:%Y-%m-%d %H} -> {stamps[-1]:%Y-%m-%d %H}")
    missing = set(pd.date_range(stamps[0], stamps[-1], freq="h")) - set(stamps)
    print(f"      {len(missing)} hours in that span have no frame")

    ds = DS[kind]
    snd_t = [pd.Timestamp(t) for t, ok in zip(ds["time"].values,
                                              np.isfinite(ds["T_snd"].values).any(axis=1)) if ok]
    drawn = sorted(set(snd_t) & set(stamps))
    print(f"      {len(drawn)} frames include a radiosonde: "
          f"{', '.join(f'{t:%m-%d %H}' for t in drawn) if drawn else '(none)'}")